In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import skimage as sk
from sklearn import mixture
import pandas as pd

%matplotlib inline

In [ ]:
channels = ['mCherry-Penta', 'GFP-Penta']
file_idx = 0

idx = 0

t_inds = list(range(0,10,1))
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        for t_ind in t_inds:
            img_ind = infile[ch][idx][t_ind]
            img = np.concatenate((img, img_ind), axis = 1)
        plt.imshow(img, cmap='gray')
        plt.show()
        
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
    seg_data = infile["data"][:]
    img = seg_data[idx][t_inds[0]]
    for t_ind in t_inds:
        img_ind = seg_data[idx][t_ind]
        img = np.concatenate((img, img_ind), axis = 1)
    plt.imshow(img)
    plt.show()

In [ ]:
nan_thr = 9 #3
percentile = 50
t_min = 0
t_max = 120


file_idx = 100

with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
        gfp_data = infile["GFP-Penta"][:]
        
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
        seg_data = infile["data"][:]
    
idx = 0 #48
t = 11

plt.imshow(seg_data[idx][t])
plt.show()
plt.imshow(gfp_data[idx][t])
plt.show()

rps = sk.measure.regionprops(seg_data[idx][t], gfp_data[idx][t])

degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
centroids = [rp.centroid for rp in rps]

# Do it for the first one
i = 4
degree = degrees[i]
centroid = centroids[i]
masked_img = (seg_data[idx][t] == (i+1))*gfp_data[idx][t]
plt.imshow(masked_img)
plt.title('masked_img')
plt.show()

print(degree)
rotated_img = sk.transform.rotate(masked_img,-degree,center=centroid[::-1],preserve_range=True)
plt.imshow(rotated_img)
plt.title('Rotated img')
plt.show()
rotated_img[rotated_img==0.]=np.NaN

n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
# plt.plot(n_not_nan, 'o')
# plt.show()
nan_mask = n_not_nan>nan_thr
nan_masked_rotated_img = rotated_img[nan_mask]

percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
sum_trace = np.nansum(nan_masked_rotated_img, axis = 1)
# percentile_traces.append(percentile_trace)
plt.plot(percentile_trace)
plt.title('median')
plt.show()
# plt.plot(sum_trace)
# plt.title('sum')

In [ ]:

f_scale = 10 # scale factor for making percentile trace integer
percentile_trace_zero_baseline = percentile_trace-min(percentile_trace)
percentile_trace_int = (percentile_trace_zero_baseline)*10
percentile_trace_int = percentile_trace_int.astype(int)

plt.plot(percentile_trace_zero_baseline)
plt.show()

trace_hist = []
for i in range(len(percentile_trace_zero_baseline)):
    # print(i)
    list_i = [i]*percentile_trace_int[i]
    # print(list_i, len(list_i))
    trace_hist += list_i
    
_ = plt.hist(trace_hist, bins=len(percentile_trace_zero_baseline))

In [ ]:
# np.sum(percentile_trace_int)
# list_all_2 = np.empty(np.sum(percentile_trace_int), dtype=int)
# for i in range(len(

In [ ]:
import scipy as sp
rolling_signal = percentile_trace - sk.restoration.rolling_ball(percentile_trace,radius=5)
med_signal = np.clip(percentile_trace - sp.signal.medfilt(percentile_trace,kernel_size=21),0,10000000)
plt.plot(rolling_signal)
plt.plot(med_signal)

In [ ]:
n_components = 5
alpha = 1.

X = np.array(trace_hist).reshape(-1,1)
gmm = mixture.BayesianGaussianMixture(n_components=n_components, 
                                      init_params='kmeans', 
                                      covariance_type="spherical", 
                                      max_iter=1000, 
                                      tol=0.01,
                                      weight_concentration_prior=1/(alpha*n_components)).fit(X)

In [ ]:
pd.DataFrame(np.stack((gmm.means_[:,0], gmm.covariances_, gmm.weights_), axis=1), columns=('Means', 'Variances', 'Weights'))

In [ ]:
ratio = gmm.weights_/gmm.covariances_

In [ ]:
ratio

In [ ]:
ratio/np.max(ratio)

In [ ]:
x = np.arange(len(percentile_trace_zero_baseline))
for i in range(n_components):
    mu = gmm.means_[i]
    sigma = np.sqrt(gmm.covariances_[i])
    plt.plot(x, sp.stats.norm.pdf(x, mu, sigma), label = str(i))
    plt.legend()

In [ ]:
prob_arr = gmm.predict_proba(np.arange(len(percentile_trace_zero_baseline)).reshape(-1,1))
# split the intensities into each Gaussian
weighted_intensity_assign = prob_arr*percentile_trace_zero_baseline[:,None]
weighted_intensity_assign_sum = np.sum(weighted_intensity_assign, axis=0)

weighted_intensity_density = weighted_intensity_assign_sum/gmm.covariances_
weighted_intensity_density

In [ ]:
for i in range(weighted_intensity_assign.shape[1]):
    plt.plot(weighted_intensity_assign[:,i], label = str(i))
    plt.legend()

In [ ]:
# Here we should have selected all of them 
density_cutoff = 70
ring_inds = np.where(weighted_intensity_density> density_cutoff)[0]

In [ ]:
for i, ring_ind in enumerate(ring_inds):
    ring_profile = weighted_intensity_assign[:,ring_ind]
    
    plt.plot(ring_profile)
    
plt.plot(percentile_trace_zero_baseline, 'k--')

In [ ]:
pk = weighted_intensity_assign[:,2]/sum(weighted_intensity_assign[:,2])
xk = np.arange(len(weighted_intensity_assign))
pmf_trace = sp.stats.rv_discrete(name='pmf', values=(xk, pk))

In [ ]:
pmf_trace.mean()

In [ ]:
pmf_trace.var()

In [ ]:
def find_width_fwhm(trace ,fwhm_tol = 0.25):
    fwhm = np.where(trace>(np.max(trace)*fwhm_tol))[0]
    fwhm = np.max(fwhm)-np.min(fwhm)
    return fwhm

find_width_fwhm(weighted_intensity_assign[:,2])

In [ ]:
def find_width_perc_cutoff(pmf_trace, percentile = 0.05):
    perc_min = np.where(pmf_trace.qvals>percentile)[0][0]
    perc_max = np.where(pmf_trace.qvals<(1.-percentile))[0][-1]
    return perc_max - perc_min

In [ ]:
find_width_perc_cutoff(pmf_trace)

## Write functions to extract data from a cell

In [ ]:
nan_thr = 3 #3
percentile = 50

file_idx = 100

with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
        seg_data = infile["data"][:]

with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
        gfp_data = infile["GFP-Penta"][:]
        
def get_intensity_projection(seg_data, gfp_data, file_idx, idx_list, t_list, nan_thr = 9, percentile = 50):
    
    columns = ('file_idx', 'trench_idx', 't', 'cell_idx', 'trace_vals', 'trace_min')
    zdf = pd.DataFrame(columns=columns)
    
    for idx in idx_list:
        for t in t_list:
            
            rps = sk.measure.regionprops(seg_data[idx][t], gfp_data[idx][t])

            degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
            centroids = [rp.centroid for rp in rps]

           
            for i, degree in enumerate(degrees):

                centroid = centroids[i]
                masked_img = (seg_data[idx][t] == (i+1))*gfp_data[idx][t]
                rotated_img = sk.transform.rotate(masked_img,-degree,center=centroid[::-1],preserve_range=True)
                rotated_img[rotated_img==0.]=np.NaN

                n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
                nan_mask = n_not_nan>nan_thr
                nan_masked_rotated_img = rotated_img[nan_mask]

                percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
                trace_min = np.min(percentile_trace)                                 
                percentile_trace_zero_baseline = percentile_trace-trace_min
                                                  
                zdf.loc[len(zdf)] = [file_idx, idx, t, i, percentile_trace_zero_baseline, trace_min]
    return zdf

In [ ]:
nan_thr = 9
percentile = 50
file_idx = 100
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
        seg_data = infile["data"][:]
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
        gfp_data = infile["GFP-Penta"][:]
        
zdf = get_intensity_projection(seg_data=seg_data, 
                         gfp_data=gfp_data,
                         file_idx=file_idx,
                         idx_list = [1],
                         t_list = [7,8,9,10,11],
                         nan_thr=nan_thr,
                         percentile=percentile)

In [ ]:
def get_gmm_data_single(trace_vals, int_scale = 10, n_components = 5, alpha = 1):
    trace_int = trace_vals*10
    trance_int = trace_int.astype(int)
    
    trace_hist = []
    for i in range(len(percentile_trace_zero_baseline)):
        list_i = [i]*percentile_trace_int[i]
        trace_hist += list_i
        
    X = np.array(trace_hist).reshape(-1,1)
    gmm = mixture.BayesianGaussianMixture(n_components=n_components, 
                                          init_params='kmeans', 
                                          covariance_type="spherical", 
                                          max_iter=1000, 
                                          tol=0.01,
                                          weight_concentration_prior=1/(alpha*n_components)).fit(X)
    
    prob_arr = gmm.predict_proba(np.arange(len(trace_vals)).reshape(-1,1))
    # df_stats = pd.DataFrame(np.stack((gmm.means_[:,0], gmm.covariances_, gmm.weights_), axis=1), columns=('Means', 'Variances', 'Weights'))
    return prob_arr, gmm.means_[:,0], gmm.covariances_, gmm.weights_

def process_single_series(z_ds):
    prob_arr, z_ds['means_gmm'], z_ds['covars_gmm'], z_ds['weights_gmm'] = get_gmm_data_single(z_ds['trace_vals'])
    # split the intensities into each Gaussian
    weighted_intensity_assign = prob_arr*trace_vals[:,None]
    weighted_intensity_assign_sum = np.sum(weighted_intensity_assign, axis=0)

    weighted_intensity_density = weighted_intensity_assign_sum/gmm.covariances_
    weighted_intensity_density
    
    z_ds['weighted_intensities'] = weighted_intensity_assign
    z_ds['densities'] = weighted_intensity_density
    
    return z_ds

In [ ]:
trace_vals = zdf.loc[0]['trace_vals']

In [ ]:
zdf.loc[0]

In [ ]:
process_single_series(zdf.loc[0])

In [ ]:
prob_arr, gmm_stats = get_gmm_data_single(zdf.loc[0]['trace_vals'])

In [ ]:
# split the intensities into each Gaussian
weighted_intensity_assign = prob_arr*trace_vals[:,None]
weighted_intensity_assign_sum = np.sum(weighted_intensity_assign, axis=0)

weighted_intensity_density = weighted_intensity_assign_sum/gmm.covariances_
weighted_intensity_density

In [ ]:
wid

In [ ]:
for i in range(weighted_intensity_assign.shape[1]):
    plt.plot(weighted_intensity_assign[:,i], label = str(i))
    plt.legend()
plt.plot(trace_vals, 'k--')

In [ ]:
plt.plot(

In [ ]:
def get_gmm_data_multi

In [ ]:
columns = ('file_idx', 'trench_idx', 't', 'cell_idx', 'trace_vals', 'trace_min', 'gmm_means', 'gmm_variances', 'gmm_weights')
zdf = pd.DataFrame(columns=columns)

In [ ]:
zdf.loc[len(zdf)] = [0, 1, 2, 3, np.array([4,5,6]), 0, 23, 24, 25]

In [ ]:
a = [0, 1, 2, 3, np.array([4,5,6]), 0, 23, 24, 25]

In [ ]:
a

In [ ]:
zdf

In [ ]:
nan_thr = 9 #3
percentile = 50
t_min = 0
t_max = 120


file_idx = 100

with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
        gfp_data = infile["GFP-Penta"][:]
        
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
        seg_data = infile["data"][:]
    
idx = 0 #48
t = 11

plt.imshow(seg_data[idx][t])
plt.show()
plt.imshow(gfp_data[idx][t])
plt.show()

rps = sk.measure.regionprops(seg_data[idx][t], gfp_data[idx][t])

degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
centroids = [rp.centroid for rp in rps]

# Do it for the first one
i = 4
degree = degrees[i]
centroid = centroids[i]
masked_img = (seg_data[idx][t] == (i+1))*gfp_data[idx][t]
plt.imshow(masked_img)
plt.title('masked_img')
plt.show()

print(degree)
rotated_img = sk.transform.rotate(masked_img,-degree,center=centroid[::-1],preserve_range=True)
plt.imshow(rotated_img)
plt.title('Rotated img')
plt.show()
rotated_img[rotated_img==0.]=np.NaN

n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
# plt.plot(n_not_nan, 'o')
# plt.show()
nan_mask = n_not_nan>nan_thr
nan_masked_rotated_img = rotated_img[nan_mask]

percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
sum_trace = np.nansum(nan_masked_rotated_img, axis = 1)
# percentile_traces.append(percentile_trace)
plt.plot(percentile_trace)
plt.title('median')
plt.show()
# plt.plot(sum_trace)
# plt.title('sum')

# Nucleoid analysis

In [ ]:
file_idx = 2
idx = 0
t = 200

dfy_single = df_yfp_test[(df_yfp_test['File Index'] == file_idx) & (df_yfp_test['File Trench Index'] == idx) & (df_yfp_test['timepoints'] == t)]
dfm_single = df_mch_test[(df_mch_test['File Index'] == file_idx) & (df_mch_test['File Trench Index'] == idx) & (df_mch_test['timepoints'] == t)]

In [ ]:
dfy_single

In [ ]:
dfy_single.columns

In [ ]:
# Vectorized for a single timepoint
bbox_single = dfy_single.loc[:, ('Global CellID', 'bbox_min_row', 'bbox_min_col', 'bbox_max_row', 'bbox_max_col')]

In [ ]:
bbox_single = bbox_single.set_index('Global CellID')

In [ ]:
bbox_single.to_xarray()

In [ ]:
nuc_single = dfm_single.loc[:,('centroid_x','centroid_y')]

In [ ]:
nuc_single

In [ ]:
nuc_single_np = nuc_single.to_numpy()
bbox_single_np = bbox_single.to_numpy()

In [ ]:
# Is x greater than min_column?
x_g_min_col = nuc_single_np[:,0][:,None] >= bbox_single_np[:,1][None,:]
print(x_g_min_col.shape)
# Is x less than max_column
x_l_max_col = nuc_single_np[:,0][:,None] <= bbox_single_np[:,3][None,:]
print(x_g_min_col.shape)

# Is y greater than min_row
y_g_min_row = nuc_single_np[:,1][:,None] >= bbox_single_np[:,0][None,:]
print(x_g_min_col.shape)
y_l_max_row = nuc_single_np[:,1][:,None] <= bbox_single_np[:,2][None,:]
print(x_g_min_col.shape)

In [ ]:
a = x_g_min_col & x_l_max_col & y_g_min_row & y_l_max_row
print( a )
indices = np.where(a)[1]

In [ ]:
indices
# Maybe add validation that it should all sum to 1 over cell axis

In [ ]:



with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
    seg_data = infile["data"][:]
    img = seg_data[idx][t_inds[0]]
    if len(t_inds) > 1:
        for t_ind in t_inds[1:]:
            img_ind = seg_data[idx][t_ind]
            img = np.concatenate((img, img_ind), axis = 1)
        
fig, ax = plt.subplots()
ax.imshow(img)

for i in range(len(dfy_single)):
    row = dfy_single.iloc[i]
    ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')
    
    n_width = row['n_bbox_max_col']-row['n_bbox_min_col']
    n_height = row['n_bbox_max_row']-row['n_bbox_min_row']
    
    
    rect = patches.Rectangle(
        (row['n_bbox_min_col']/pixel_microns, row['n_bbox_min_row']/pixel_microns), 
        n_width/pixel_microns,
        n_height/pixel_microns,
        facecolor = 'none', edgecolor='w', linewidth = 1)
    ax.add_patch(rect)
    
plt.show()

In [ ]:
# Plot nucleoids with bounding boxes

channels = ['mCherry']
t_inds = [t]
with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        if len(t_inds) > 1:
            for t_ind in t_inds[1:]:
                img_ind = infile[ch][idx][t_ind]
                img = np.concatenate((img, img_ind), axis = 1)

fig, ax = plt.subplots()
ax.imshow(img, cmap='gray')

for i in range(len(dfy_single)):
    row = dfy_single.iloc[i]
    # ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')
    
    n_width = row['n_bbox_max_col']-row['n_bbox_min_col']
    n_height = row['n_bbox_max_row']-row['n_bbox_min_row']
    
    ax.plot(row['bbox_min_col']/pixel_microns, row['bbox_min_row']/pixel_microns, 'o')
    
    rect = patches.Rectangle(
        (row['n_bbox_min_col']/pixel_microns, row['n_bbox_min_row']/pixel_microns), 
        n_width/pixel_microns,
        n_height/pixel_microns,
        facecolor = 'none', edgecolor='w', linewidth = 1)
    ax.add_patch(rect)
    
for i in range(len(dfm_single)):
    plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o', color = colors[cell_assign[i]%len(colors)])
    
plt.show()


with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
    seg_data = infile["data"][:]
    img = seg_data[idx][t_inds[0]]
    if len(t_inds) > 1:
        for t_ind in t_inds[1:]:
            img_ind = seg_data[idx][t_ind]
            img = np.concatenate((img, img_ind), axis = 1)
        
fig, ax = plt.subplots()
ax.imshow(img)

for i in range(len(dfy_single)):
    row = dfy_single.iloc[i]
    ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')
    
    n_width = row['n_bbox_max_col']-row['n_bbox_min_col']
    n_height = row['n_bbox_max_row']-row['n_bbox_min_row']
    
    
    rect = patches.Rectangle(
        (row['n_bbox_min_col']/pixel_microns, row['n_bbox_min_row']/pixel_microns), 
        n_width/pixel_microns,
        n_height/pixel_microns,
        facecolor = 'none', edgecolor='w', linewidth = 1)
    ax.add_patch(rect)
    
plt.show()

In [ ]:


channels = ['mCherry']
t_inds = [t]
with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        if len(t_inds) > 1:
            for t_ind in t_inds[1:]:
                img_ind = infile[ch][idx][t_ind]
                img = np.concatenate((img, img_ind), axis = 1)

fig, ax = plt.subplots()
ax.imshow(img, cmap='gray')

for i in range(len(dfy_single)):
    row = dfy_single.iloc[i]
    ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')
    
    n_width = row['n_bbox_max_col']-row['n_bbox_min_col']
    n_height = row['n_bbox_max_row']-row['n_bbox_min_row']
    
    
    rect = patches.Rectangle(
        (row['n_bbox_min_col']/pixel_microns, row['n_bbox_min_row']/pixel_microns), 
        n_width/pixel_microns,
        n_height/pixel_microns,
        facecolor = 'none', edgecolor='w', linewidth = 1)
    ax.add_patch(rect)
    
plt.show()


with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
    seg_data = infile["data"][:]
    img = seg_data[idx][t_inds[0]]
    if len(t_inds) > 1:
        for t_ind in t_inds[1:]:
            img_ind = seg_data[idx][t_ind]
            img = np.concatenate((img, img_ind), axis = 1)
        
fig, ax = plt.subplots()
ax.imshow(img)

for i in range(len(dfy_single)):
    row = dfy_single.iloc[i]
    ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')
    
    n_width = row['n_bbox_max_col']-row['n_bbox_min_col']
    n_height = row['n_bbox_max_row']-row['n_bbox_min_row']
    
    
    rect = patches.Rectangle(
        (row['n_bbox_min_col']/pixel_microns, row['n_bbox_min_row']/pixel_microns), 
        n_width/pixel_microns,
        n_height/pixel_microns,
        facecolor = 'none', edgecolor='w', linewidth = 1)
    ax.add_patch(rect)
    
plt.show()

In [ ]:
channels = ['mCherry']
with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        if len(t_inds) > 1:
            for t_ind in t_inds[1:]:
                img_ind = infile[ch][idx][t_ind]
                img = np.concatenate((img, img_ind), axis = 1)
                
plt.imshow(img, cmap='gray')

for i in range(len(dfm_single)):
    plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o')
    
plt.show()

        
with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
    seg_data = infile["data"][:]
    img = seg_data[idx][t_inds[0]]
    if len(t_inds) > 1:
        for t_ind in t_inds[1:]:
            img_ind = seg_data[idx][t_ind]
            img = np.concatenate((img, img_ind), axis = 1)
        
fig, ax = plt.subplots()
ax.imshow(img)

for i in range(len(dfy_single)):
    row = dfy_single.iloc[i]
    ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')
    
    n_width = row['n_bbox_max_col']-row['n_bbox_min_col']
    n_height = row['n_bbox_max_row']-row['n_bbox_min_row']
    
    
    rect = patches.Rectangle(
        (row['n_bbox_min_col']/pixel_microns, row['n_bbox_min_row']/pixel_microns), 
        n_width/pixel_microns,
        n_height/pixel_microns,
        facecolor = 'none', edgecolor='w', linewidth = 1)
    ax.add_patch(rect)
    
plt.show()

In [ ]:
channels = ['mCherry', 'YFP']
file_idx = 0
idx = 20

t_inds = list(range(0,100,1))
with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        for t_ind in t_inds:
            img_ind = infile[ch][idx][t_ind]
            img = np.concatenate((img, img_ind), axis = 1)
        plt.imshow(img, cmap='gray')
        plt.show()
        
with h5py.File("/home/lag36/scratch/lag36/2022-10-03_lDE25_From_2022-08-24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
    seg_data = infile["data"][:]
    img = seg_data[idx][t_inds[0]]
    for t_ind in t_inds:
        img_ind = seg_data[idx][t_ind]
        img = np.concatenate((img, img_ind), axis = 1)
    plt.imshow(img)
    plt.show()

In [ ]:
df_yfp_test.loc[:,('bbox_min_row', 'bbox_min_col', 'bbox_max_row', 'bbox_max_col')].to_numpy()
df_yfp_test.groupby('Kymograph File Parquet Index').apply(lambda df:df.loc[:,('bbox_min_row', 'bbox_min_col', 'bbox_max_row', 'bbox_max_col')].to_numpy())